# CSV: Encodings, Dialects, Corrupt Rows, Combining Files

## What you'll practice

The `csv` module handles quoting, embedded commas and newlines — never `line.split(",")`. Real files add BOMs, different delimiters,
reordered columns, extra columns and ragged rows. You'll handle all of them using the files in `datasets/incoming/`.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/files/test_06_02_csv_processing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/06_files_json_csv/02_csv_processing.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
INCOMING = DATA / "incoming"
print(sorted(p.name for p in INCOMING.iterdir()))

## Exercise 1 — 🟢 Easy · Read CSV robustly (BOM + header cleanup)

**Implement:** `read_csv(path) -> list[dict]`

Read a CSV into dicts. Use `encoding="utf-8-sig"` (strips a BOM if present) and `newline=""`. Strip whitespace from header names. `orders_03.csv` has a BOM — its first key must be `order_ts`, not `\ufefforder_ts`.

**Input:** `path`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_read_csv_bom_and_header, test_read_csv_real_bom` in `tests/files/test_06_02_csv_processing.py`.

**Check:** `pytest tests/files/test_06_02_csv_processing.py -k "test_read_csv_bom_and_header or test_read_csv_real_bom"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Write CSV with inferred columns

**Implement:** `write_csv(path, rows: list[dict], fieldnames: list[str] | None = None) -> int`

Write rows to CSV. If `fieldnames` is `None`, use the union of keys in first-seen order. Missing values → empty string; with explicit `fieldnames`,
extra keys are ignored (`extrasaction="ignore"`). Create parent directories. Return rows written. Values `None` should be written as empty.

**Input:** `path, rows: list[dict], fieldnames: list[str] | None = None`

**Expected output:** `int`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_write_csv, test_write_csv_explicit_fields` in `tests/files/test_06_02_csv_processing.py`.

**Check:** `pytest tests/files/test_06_02_csv_processing.py -k "test_write_csv or test_write_csv_explicit_fields"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Detect the delimiter

**Implement:** `read_any_delimited(path) -> tuple[str, list[dict]]`

Vendors send `,`, `;`, `|` or tab-separated files. Use `csv.Sniffer().sniff(sample, delimiters=",;|\t")` on the first ~4 KB, then read with that dialect. Return `(delimiter, rows)`. If sniffing fails, fall back to `,`.

**Input:** `path`

**Expected output:** `tuple[str, list[dict]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_read_any_delimited` in `tests/files/test_06_02_csv_processing.py`.

**Check:** `pytest tests/files/test_06_02_csv_processing.py -k test_read_any_delimited`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Quarantine corrupt rows with line numbers

**Implement:** `split_good_bad(path, required: list[str], numeric: list[str]) -> tuple[list[dict], list[dict]]`

Read with `csv.reader` (not DictReader) so you can see ragged rows. Skip completely blank lines. For each data row, produce a bad-record entry
`{"line": reader.line_num, "reason": ..., "raw": row_list}` when: the field count differs from the header (`"expected N fields, got M"`), a required field is
blank (`"missing <field>"`), or a numeric field doesn't parse as float (`"non-numeric <field>"`). First problem wins. Good rows become dicts.
Try it on `datasets/incoming/corrupt.csv` — and look closely at what the line with an unbalanced `"` does to the line after it.

**Input:** `path, required: list[str], numeric: list[str]`

**Expected output:** `tuple[list[dict], list[dict]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_split_good_bad, test_split_good_bad_real_corrupt` in `tests/files/test_06_02_csv_processing.py`.

**Check:** `pytest tests/files/test_06_02_csv_processing.py -k "test_split_good_bad or test_split_good_bad_real_corrupt"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Combine heterogeneous incoming files · 🧩 Mini-project

**Implement:** `combine_incoming(incoming_dir, output_path, columns: list[str]) -> dict`

Combine every `orders_*.csv` in `incoming_dir` (use your discovery rules: no `.tmp`, hidden or `_` files) into one CSV with the canonical `columns` order,
followed by any **extra** columns seen (first-seen order), followed by a `source_file` column (file name). Handle BOMs and reordered columns. Rows with the
wrong field count are skipped and counted. Stream rows — don't hold all files in memory (you may do two passes: one over headers, one over rows).
Return `{"files": [names...], "rows_written": n, "rows_skipped": n, "extra_columns": [...]}`.

**Input:** `incoming_dir, output_path, columns: list[str]`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
COLS = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "order_ts", "status", "updated_at", "channel", "country"]
combine_incoming("../datasets/incoming", "../output/combined_orders.csv", COLS)
```

**Check:** `pytest tests/files/test_06_02_csv_processing.py -k "test_combine_incoming or test_combine_incoming_real"`

**YOUR CODE ↓**